# VisDSR: Qwen3-VL-8B on Kaggle

Enable a Kaggle GPU and Internet. This notebook now defaults to the three-call `pilot2_smoke` check for the second and final calibration round. Attach the completed first-pilot `visdsr_results.zip` as a Kaggle input before Run All. Review the new smoke output, then set `RUN_MODE = "pilot2"` and `PILOT2_SMOKE_REVIEWED = True` for the remaining calls. The main run still requires a separate freeze decision and `MAIN_APPROVED = True`.

Model weights come from the [official Qwen checkpoint](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct). The repository provides the tasks, prompt, images, parser, and scoring. No inference API is called.


In [ ]:
RUN_MODE = "pilot2_smoke"  # smoke, pilot, pilot2_smoke, pilot2, or main
SMOKE_REVIEWED = False
PILOT2_SMOKE_REVIEWED = False
MAIN_APPROVED = False
REPOSITORY = "https://github.com/Abhishekgit01/VisDSR.git"
# Keep REPOSITORY as a plain URL, without Markdown brackets.


## Locate the project

An attached Kaggle Dataset containing the project is used first. Otherwise the notebook clones the public repository. Attach the completed first-pilot `visdsr_results.zip` as a Kaggle input. The notebook restores only its data and results, keeping the current repository configuration. Download the new export before leaving this session.


In [ ]:
from pathlib import Path
import shutil
import socket
import subprocess
import sys
import zipfile

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ROOT = WORK / "VisDSR"

def run_command(*parts, cwd=None):
    subprocess.run(parts, cwd=cwd or ROOT, check=True)

def safe_extract(archive, destination, prefixes=None):
    with zipfile.ZipFile(archive) as source:
        selected = [member for member in source.infolist()
                    if prefixes is None or member.filename.startswith(prefixes)]
        for member in selected:
            path = (destination / member.filename).resolve()
            if not path.is_relative_to(destination.resolve()):
                raise ValueError(f"unsafe archive path: {member.filename}")
        source.extractall(destination, members=selected)

if not (ROOT / "configs/experiment.yaml").exists():
    candidates = list(Path("/kaggle/input").rglob("configs/experiment.yaml")) if Path("/kaggle/input").exists() else []
    candidates = [p for p in candidates if (p.parent.parent / "sim").exists()]
    if candidates:
        source = candidates[0].parent.parent
        shutil.copytree(source, ROOT, dirs_exist_ok=True)
        print("Copied project from Kaggle input:", source)
    else:
        archives = list(Path("/kaggle/input").rglob("*.zip")) if Path("/kaggle/input").exists() else []
        project_archive = next((p for p in archives if "visdsr" in p.name.lower() and "results" not in p.name.lower()), None)
        if project_archive:
            safe_extract(project_archive, WORK)
            extracted = [p.parent.parent for p in WORK.rglob("configs/experiment.yaml")
                         if (p.parent.parent / "sim").exists()]
            if extracted and extracted[0] != ROOT:
                shutil.copytree(extracted[0], ROOT, dirs_exist_ok=True)
            print("Extracted project archive:", project_archive)
        else:
            if not REPOSITORY.startswith("https://github.com/") or any(char in REPOSITORY for char in "[]()"):
                raise ValueError("REPOSITORY must be a plain GitHub URL, without Markdown brackets")
            try:
                socket.getaddrinfo("github.com", 443)
            except OSError as exc:
                raise RuntimeError(
                    "Kaggle cannot resolve github.com. In the notebook's right-side "
                    "Session options, turn Internet ON, restart the session, and Run All again. "
                    "Internet is also needed later for packages and model weights."
                ) from exc
            run_command("git", "clone", "--depth", "1", REPOSITORY, str(ROOT), cwd=WORK)
            print("Cloned project:", REPOSITORY)
if not (ROOT / "configs/experiment.yaml").exists():
    raise FileNotFoundError("VisDSR project not found; attach it as a Kaggle Dataset or enable Internet")
for folder in ("sim", "gen", "eval", "analysis", "configs"):
    if not (ROOT / folder).exists():
        raise FileNotFoundError(f"missing project folder: {folder}")
(ROOT / "data").mkdir(exist_ok=True)
(ROOT / "results").mkdir(exist_ok=True)
print("Project root:", ROOT)

if Path("/kaggle/input").exists():
    previous = list(Path("/kaggle/input").rglob("visdsr_results.zip"))
    if previous:
        safe_extract(previous[0], ROOT, prefixes=("data/", "results/"))
        print("Restored previous results:", previous[0])
    else:
        snapshots = list(Path("/kaggle/input").rglob("cache_snapshot.zip"))
        if snapshots:
            safe_extract(snapshots[0], ROOT)
            print("Restored previous cache:", snapshots[0])


## Install dependencies and report the runtime

The notebook uses Kaggle's installed PyTorch and CUDA build. It installs pinned Transformers 4.57.1 and the 4-bit loading packages while keeping installed dependencies where compatible.


In [ ]:
import platform
print("Python:", platform.python_version())
try:
    import torch
except ImportError as exc:
    raise RuntimeError("Kaggle's PyTorch installation is required") from exc
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before Run All")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name}; VRAM {props.total_memory / 2**30:.2f} GiB")

run_command(sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "environment/requirements.txt"))
run_command(sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
            "transformers==4.57.1", "accelerate", "bitsandbytes", "huggingface_hub")
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
import transformers
import yaml
settings = yaml.safe_load((ROOT / "configs/experiment.yaml").read_text())["models"][0]
print("Transformers:", transformers.__version__)
print("Model ID:", settings["id"])
print("Pinned revision:", settings["revision"])
print("Generation settings:", {k: settings[k] for k in ("do_sample", "num_beams", "max_output_tokens", "quantization")})


## Build and validate the selected input

The original four-operation pilot remains untouched. `pilot2` has 12 new tasks: three per combination of 8 or 16 elements and one or two operations. The same renderers, prompts, and scorer are used. No individual image upload is needed.


In [ ]:
fonts = [Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
         Path("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf")]
if shutil.which("dot") is None or not all(path.exists() for path in fonts):
    run_command("apt-get", "update", "-qq")
    run_command("apt-get", "install", "-y", "-qq", "graphviz", "fonts-dejavu-core")
run_command("make", "build")
input_split = "pilot2" if RUN_MODE in ("pilot2_smoke", "pilot2") else "pilot"
if input_split == "pilot2":
    if not (ROOT / "results/scores_pilot_model1.csv").exists():
        raise RuntimeError("Attach the completed first-pilot visdsr_results.zip as a Kaggle input before pilot2")
    if not (ROOT / "data/pilot/tasks.jsonl").exists():
        raise RuntimeError("The first-pilot tasks are missing from the attached results archive")
    run_command(sys.executable, "-m", "gen.validate", "--split", "pilot")
if not (ROOT / "data" / input_split / "tasks.jsonl").exists():
    run_command(sys.executable, "-m", "gen.generate", "--split", input_split, "--structure", "dsu")
if not (ROOT / "data" / input_split / "manifest.csv").exists():
    run_command(sys.executable, "-m", "gen.render", "--split", input_split)
run_command(sys.executable, "-m", "gen.validate", "--split", input_split)
print(input_split, "tasks and images ready")


## Run the selected mode

`pilot2_smoke` makes exactly three new calibration calls (T-dir, R-dir, G-dir) and stops for review. `pilot2` completes all 60 calls after its smoke report is reviewed. Every response is cached immediately; reruns skip saved calls. The official strict scoring is unchanged. The separate diagnostic report never changes official scores or decides the freeze.


In [ ]:
if RUN_MODE == "smoke":
    run_command(sys.executable, "-m", "eval.run", "--split", "pilot", "--model", "model1", "--smoke")
    print("STOP: review this smoke report before enabling pilot mode.")
elif RUN_MODE == "pilot":
    if not SMOKE_REVIEWED:
        raise RuntimeError("Review the smoke report, then set SMOKE_REVIEWED = True")
    run_command(sys.executable, "-m", "eval.run", "--split", "pilot", "--model", "model1", "--after-smoke-review")
    run_command(sys.executable, "-m", "analysis.analyze", "--split", "pilot", "--models", "model1")
elif RUN_MODE == "pilot2_smoke":
    run_command(sys.executable, "-m", "eval.run", "--split", "pilot2", "--model", "model1", "--smoke")
    print("STOP: review this three-call pilot2 report before the full calibration run.")
elif RUN_MODE == "pilot2":
    if not PILOT2_SMOKE_REVIEWED:
        raise RuntimeError("Review the pilot2 smoke report, then set PILOT2_SMOKE_REVIEWED = True")
    run_command(sys.executable, "-m", "eval.run", "--split", "pilot2", "--model", "model1", "--after-smoke-review")
    run_command(sys.executable, "-m", "analysis.analyze", "--split", "pilot2", "--models", "model1")
    run_command(sys.executable, "-m", "analysis.pilot2", "--model", "model1")
elif RUN_MODE == "main":
    if not (SMOKE_REVIEWED and MAIN_APPROVED):
        raise RuntimeError("Main requires reviewed smoke and explicit main approval")
    if not (ROOT / "FREEZE.json").exists():
        raise RuntimeError("Freeze the protocol before generating main tasks")
    if not (ROOT / "data/main/tasks.jsonl").exists():
        run_command(sys.executable, "-m", "gen.generate", "--split", "main", "--structure", "dsu")
    if not (ROOT / "data/main/manifest.csv").exists():
        run_command(sys.executable, "-m", "gen.render", "--split", "main")
    run_command(sys.executable, "-m", "gen.validate", "--split", "main")
    run_command(sys.executable, "-m", "eval.run", "--split", "main", "--model", "model1", "--after-smoke-review")
    run_command(sys.executable, "-m", "analysis.analyze", "--split", "main", "--models", "model1")
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")


In [ ]:
if RUN_MODE == "pilot":
    import csv
    from collections import Counter
    with (ROOT / "results/scores_pilot_model1.csv").open(newline="") as source:
        scored = list(csv.DictReader(source))
    if len(scored) != 60:
        raise RuntimeError(f"Expected 60 pilot results; found {len(scored)}")
    print("PILOT SUMMARY (12 tasks per condition)")
    print("condition  final  step  sequence  transcription  format errors")
    accuracies = {}
    for condition in ("T-dir", "R-dir", "G-dir", "T-str", "G-str"):
        subset = [row for row in scored if row["condition"] == condition]
        final = sum(int(row["final_correct"]) for row in subset) / len(subset)
        step = sum(float(row["per_step_accuracy"]) for row in subset) / len(subset)
        sequence = sum(int(row["full_sequence_correct"]) for row in subset) / len(subset)
        transcribed = [int(row["transcription_correct"]) for row in subset
                       if row["transcription_correct"] != ""]
        transcription = f"{sum(transcribed) / len(transcribed):.2%}" if transcribed else "--"
        errors = sum(int(row["format_error"]) for row in subset)
        accuracies[condition] = final
        print(f"{condition:9}  {final:5.1%}  {step:5.1%}  {sequence:8.1%}  "
              f"{transcription:13}  {errors}")
    failures = Counter(row["failure_type"] for row in scored)
    print("Format errors:", failures["format_error"])
    print("Model/runtime errors: 0 (the complete run finished)")
    print("Structure-extraction failures:", failures["structure_extraction_failure"])
    print("State-transition failures:", failures["state_transition_failure"])
    print("PILOT RECOMMENDATION")
    if accuracies["T-dir"] == accuracies["G-dir"] and accuracies["T-dir"] in (0, 1):
        print("ONE CALIBRATION CHANGE RECOMMENDED: both direct conditions are at floor or ceiling; review manually.")
    else:
        print("FREEZE CURRENT DIFFICULTY, subject to manual review of images and responses.")
    print("No calibration change has been applied.")


## Export results

Run the export cell after the three-call smoke and again after the full calibration run. Download `/kaggle/working/visdsr_results.zip` before leaving Kaggle. Attach that latest archive as a Kaggle input to resume later. The archive includes both pilot rounds; the repository code, prompts, and scoring remain separate.


In [ ]:
run_command(sys.executable, "-m", "manifest")
export = WORK / "visdsr_results.zip"
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ("results", "data"):
        for path in sorted((ROOT / folder).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(ROOT))
    for name in ("MANIFEST.json", "configs/experiment.yaml", "CHANGELOG.md"):
        path = ROOT / name
        if path.exists():
            archive.write(path, name)
print("Export:", export)
